# Fase 2 Inferencia Estadística del Impacto Geomecánico

## Objetivo

Determinar estadísticamente si las diferencias de desgaste observadas entre el Tajo Norte y el Tajo Sur son significativas desde el punto de vista operacional y geomecánico.

Para ello se desarrollará:

- análisis descriptivo,
- análisis exploratorio,
- prueba de hipótesis,
- inferencia estadística,
- y visualización comparativa.

La finalidad es demostrar matemáticamente que las condiciones geomecánicas afectan directamente la vida útil de los neumáticos mineros.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 1. Carga de Datos

Se cargarán las hojas:

- 00 Estudio
- 01 Parametros

desde el archivo Excel original proporcionado para el análisis.

In [14]:
# RUTA DEL ARCHIVO
ruta_excel = "C:/Users/estud/OneDrive/Escritorio/Proyecto_LasBambas/Proyecto_LasBambas/data/raw/00 Data (12).xlsx"

# CARGA DE HOJAS
df_estudio = pd.read_excel(ruta_excel,sheet_name="00 Estudio")
df_parametros = pd.read_excel(ruta_excel,sheet_name="01 Parametros")

print("Datos cargados correctamente")
print("\nDimensiones:")
print(df_estudio.shape)

df_estudio.head()

Datos cargados correctamente

Dimensiones:
(10, 8)


,ID_Camion_Test,Tajo_Asignado,Periodo_Prueba,Horas_Trabajadas,Profundidad_Inicial_mm,Profundidad_Final_mm,Vida_Util_Proyectada_horas,Condiciones
0,TEST_001,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1424.8,65,55.8,6201,"Roca 70 MPa, +10% pendiente, sube cargado"
1,TEST_002,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1393.1,65,56.0,6201,"Roca 70 MPa, +10% pendiente, sube cargado"
2,TEST_003,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1432.4,65,55.8,6201,"Roca 70 MPa, +10% pendiente, sube cargado"
3,TEST_004,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1476.2,65,55.5,6201,"Roca 70 MPa, +10% pendiente, sube cargado"
4,TEST_005,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1388.3,65,56.0,6201,"Roca 70 MPa, +10% pendiente, sube cargado"


## 2. Ingeniería de Variables

Se calcularán las variables necesarias para el análisis estadístico:

- Desgaste acumulado.
- Tasa de desgaste.
- Costo por Hora (CPH).

Estas métricas permitirán cuantificar el impacto operacional de cada zona minera.

In [15]:
# COSTO UNITARIO LLANTA

precio_unitario = 52000
costo_total_llantas = precio_unitario * 6


# DESGASTE
df_estudio["Desgaste_mm"] = (df_estudio["Profundidad_Inicial_mm"]- df_estudio["Profundidad_Final_mm"])


# TASA DE DESGASTE
df_estudio["Tasa_Desgaste_mm_h"] = (df_estudio["Desgaste_mm"]/df_estudio["Horas_Trabajadas"])


# CPH
df_estudio["CPH_Proyectado"] = (costo_total_llantas/df_estudio["Vida_Util_Proyectada_horas"])
df_estudio.head()

,ID_Camion_Test,Tajo_Asignado,Periodo_Prueba,Horas_Trabajadas,Profundidad_Inicial_mm,Profundidad_Final_mm,Vida_Util_Proyectada_horas,Condiciones,Desgaste_mm,Tasa_Desgaste_mm_h,CPH_Proyectado
0,TEST_001,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1424.8,65,55.8,6201,"Roca 70 MPa, +10% pendiente, sube cargado",9.2,0.006457,50.314465
1,TEST_002,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1393.1,65,56.0,6201,"Roca 70 MPa, +10% pendiente, sube cargado",9.0,0.006460,50.314465
2,TEST_003,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1432.4,65,55.8,6201,"Roca 70 MPa, +10% pendiente, sube cargado",9.2,0.006423,50.314465
3,TEST_004,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1476.2,65,55.5,6201,"Roca 70 MPa, +10% pendiente, sube cargado",9.5,0.006435,50.314465
4,TEST_005,Tajo_1 (Tajo Norte),Jul-Oct 2024 (4 meses),1388.3,65,56.0,6201,"Roca 70 MPa, +10% pendiente, sube cargado",9.0,0.006483,50.314465


## 3. Estadística Descriptiva

Se realizará un análisis descriptivo de las variables principales para comprender:

- distribución,
- promedio,
- dispersión,
- y comportamiento general de los datos.

Esto permitirá identificar diferencias preliminares entre ambos tajos.

In [16]:
# RESUMEN ESTADÍSTICO

resumen = df_estudio.groupby("Tajo_Asignado")[["Tasa_Desgaste_mm_h","CPH_Proyectado"]].agg(["mean","std","min","max"])

resumen

Tasa_Desgaste_mm_h                                \
                                  mean       std       min       max   
Tajo_Asignado                                                          
Tajo_1 (Tajo Norte)           0.006452  0.000023  0.006423  0.006483   
Tajo_2 (Tajo Sur)             0.008342  0.000016  0.008316  0.008356   

                    CPH_Proyectado                             
                              mean  std        min        max  
Tajo_Asignado                                                  
Tajo_1 (Tajo Norte)      50.314465  0.0  50.314465  50.314465  
Tajo_2 (Tajo Sur)        64.986461  0.0  64.986461  64.986461

## 4. Visualización Exploratoria

Se desarrollarán visualizaciones comparativas para analizar:

- diferencias de desgaste,
- dispersión operacional,
- y comportamiento financiero entre tajos.

Las gráficas permitirán interpretar visualmente el impacto geomecánico.

In [17]:
import plotly.graph_objects as go

# DATOS

norte = df_estudio[df_estudio["Tajo_Asignado"].str.contains("Norte", case=False)]["Tasa_Desgaste_mm_h"]
sur = df_estudio[df_estudio["Tajo_Asignado"].str.contains("Sur", case=False)]["Tasa_Desgaste_mm_h"]

# DIFERENCIA %

incremento = ((sur.mean() - norte.mean())/norte.mean()) * 100

# BOXPLOT SIMPLE Y EJECUTIVO

fig = go.Figure()

# Tajo Norte
fig.add_trace(
    go.Box(
        y=norte,
        name="Tajo Norte",
        boxmean=True
    )
)

# Tajo Sur
fig.add_trace(
    go.Box(
        y=sur,
        name="Tajo Sur",
        boxmean=True
    )
)

# DISEÑO

fig.update_layout(
    title="Distribución de Desgaste por Zona Operacional",
    title_x=0.5,
    xaxis_title="Zona Minera",
    yaxis_title="Tasa de Desgaste (mm/h)",
    template="plotly_white",
    height=600,
    width=900,
    font=dict(size=14)
)

# ANOTACIÓN

fig.add_annotation(x="Tajo Sur",y=sur.mean(),text=f"{incremento:.1f}% más desgaste",showarrow=True,arrowhead=2)

# MOSTRAR
fig.show()

In [18]:
import pandas as pd
import plotly.express as px

# DATAFRAME COMPARATIVO

df_comparacion = pd.DataFrame({"Tajo": ["Tajo Norte","Tajo Sur"],"Tasa_Desgaste_mm_h": [norte.mean(),sur.mean()]})

# DIFERENCIA %
incremento = ((sur.mean()-norte.mean())/norte.mean()) * 100

# GRÁFICO

fig = px.bar(
    df_comparacion,
    x="Tajo",
    y="Tasa_Desgaste_mm_h",
    text="Tasa_Desgaste_mm_h",
    template="plotly_white",
    title="Comparación Promedio de Desgaste por Zona"
)


# PERSONALIZACIÓN

fig.update_traces(
    texttemplate="%{text:.5f} mm/h",
    textposition="outside"
)

fig.update_layout(
    title_x=0.5,
    title_font_size=24,
    xaxis_title="Zona Operacional",
    yaxis_title="Tasa de Desgaste (mm/h)",
    height=650,
    width=1000,
    font=dict(size=15)
)

# ANOTACIÓN EJECUTIVA

fig.add_annotation(
    x="Tajo Sur",
    y=sur.mean(),
    text=(f"<b>{incremento:.1f}%</b><br>"f"más desgaste"),
    showarrow=True,
    arrowhead=2,
    yshift=40,
    bordercolor="black",
    borderwidth=1,
    bgcolor="white"
)

# MOSTRAR
fig.show()

## 5. Planteamiento de Hipótesis

Para validar matemáticamente el impacto geomecánico se utilizará una prueba T de Student para muestras independientes.

### Hipótesis Nula ($H_0$)

No existen diferencias significativas en la tasa de desgaste entre el Tajo Norte y el Tajo Sur.

### Hipótesis Alternativa ($H_1$)

Sí existen diferencias significativas en la tasa de desgaste entre ambos tajos.

In [19]:
# SEPARAR MUESTRAS

tajo_norte = df_estudio[df_estudio["Tajo_Asignado"].str.contains("Norte", case=False)]
tajo_sur = df_estudio[df_estudio["Tajo_Asignado"].str.contains("Sur", case=False)]


# VARIABLES

tasa_norte = tajo_norte["Tasa_Desgaste_mm_h"]
tasa_sur = tajo_sur["Tasa_Desgaste_mm_h"]


# T-TEST DE WELCH

t_stat, p_value = stats.ttest_ind(tasa_norte,tasa_sur,equal_var=False)


# RESULTADOS

print("="*55)
print(" RESULTADOS DEL T-TEST")
print("="*55)
print(f"Media Tajo Norte: "f"{tasa_norte.mean():.5f} mm/h")
print(f"Media Tajo Sur: "f"{tasa_sur.mean():.5f} mm/h")
print("-"*55)
print(f"T-Statistic : {t_stat:.4f}")
print(f"P-Value     : {p_value:.10f}")
print("-"*55)

if p_value < 0.05:
    print("Se rechaza la Hipótesis Nula")
    print( "Existe diferencia estadísticamente significativa")
else:
    print("No se rechaza la Hipótesis Nula")

 RESULTADOS DEL T-TEST
Media Tajo Norte: 0.00645 mm/h
Media Tajo Sur: 0.00834 mm/h
-------------------------------------------------------
T-Statistic : -150.0223
P-Value     : 0.0000000000
-------------------------------------------------------
Se rechaza la Hipótesis Nula
Existe diferencia estadísticamente significativa


## 6. Interpretación Estadística

El análisis inferencial permitió demostrar matemáticamente que las condiciones geomecánicas impactan significativamente el desgaste de neumáticos.

El Tajo Sur presentó mayores niveles de desgaste debido a:

- mayor dureza de roca,
- mayores esfuerzos operacionales,
- y condiciones térmicas más severas.

Los resultados estadísticos justifican técnicamente la necesidad de implementar estrategias inteligentes de asignación dinámica.